In [ ]:
print("MVP project start")

MVP project start


In [ ]:
# ⚠️ Colab only — remove if running locally
!pip install yfinance

In [ ]:
import pandas as pd
import yfinance as yf

print("libraries loaded")

libraries loaded


In [ ]:
assets = ["BTC-USD", "ETH-USD", "SPY", "QQQ"]

all_data = []

for asset in assets:
    data = yf.download(asset, start="2023-01-01")

    df = data[["Close"]].copy()
    df.columns = ["close"]

    # 1. 日報酬率
    df["daily_return"] = df["close"].pct_change()

    # 2. 累積報酬率
    df["cum_return"] = (1 + df["daily_return"]).cumprod()
    df["cum_return"] = df["cum_return"].fillna(1)

    # 3. 30日波動度
    df["volatility_30d"] = df["daily_return"].rolling(30).std()

    # 4. 30日報酬率
    df["rolling_return_30d"] = df["close"].pct_change(30)

    # 5. 90日報酬率
    df["rolling_return_90d"] = df["close"].pct_change(90)

    # 6. 歷史最高價
    df["rolling_max"] = df["close"].cummax()

    # 7. 回撤（drawdown）
    df["drawdown"] = df["close"] / df["rolling_max"] - 1

    # 8. 資產名稱
    df["asset"] = asset

    all_data.append(df)

master = pd.concat(all_data)
master = master.reset_index()

master.head()

/tmp/ipykernel_13935/3972876651.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(asset, start="2023-01-01")
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_13935/3972876651.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(asset, start="2023-01-01")
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_13935/3972876651.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(asset, start="2023-01-01")
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_13935/3972876651.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(asset, start="2023-01-01")
[*********************100%***********************]  1 of 1 completed


,Date,close,daily_return,cum_return,volatility_30d,rolling_return_30d,rolling_return_90d,rolling_max,drawdown,asset
0,2023-01-01,16625.080078,NaN,1.000000,NaN,NaN,NaN,16625.080078,0.000000,BTC-USD
1,2023-01-02,16688.470703,0.003813,1.003813,NaN,NaN,NaN,16688.470703,0.000000,BTC-USD
2,2023-01-03,16679.857422,-0.000516,1.003295,NaN,NaN,NaN,16688.470703,-0.000516,BTC-USD
3,2023-01-04,16863.238281,0.010994,1.014325,NaN,NaN,NaN,16863.238281,0.000000,BTC-USD
4,2023-01-05,16836.736328,-0.001572,1.012731,NaN,NaN,NaN,16863.238281,-0.001572,BTC-USD


In [ ]:
master.columns

Index(['Date', 'close', 'daily_return', 'cum_return', 'volatility_30d',
       'rolling_return_30d', 'rolling_return_90d', 'rolling_max', 'drawdown',
       'asset'],
      dtype='object')

In [ ]:
master.head(10)

,Date,close,daily_return,cum_return,volatility_30d,rolling_return_30d,rolling_return_90d,rolling_max,drawdown,asset
0,2023-01-01,16625.080078,NaN,1.000000,NaN,NaN,NaN,16625.080078,0.000000,BTC-USD
1,2023-01-02,16688.470703,0.003813,1.003813,NaN,NaN,NaN,16688.470703,0.000000,BTC-USD
2,2023-01-03,16679.857422,-0.000516,1.003295,NaN,NaN,NaN,16688.470703,-0.000516,BTC-USD
3,2023-01-04,16863.238281,0.010994,1.014325,NaN,NaN,NaN,16863.238281,0.000000,BTC-USD
4,2023-01-05,16836.736328,-0.001572,1.012731,NaN,NaN,NaN,16863.238281,-0.001572,BTC-USD
5,2023-01-06,16951.968750,0.006844,1.019662,NaN,NaN,NaN,16951.968750,0.000000,BTC-USD
6,2023-01-07,16955.078125,0.000183,1.019849,NaN,NaN,NaN,16955.078125,0.000000,BTC-USD
7,2023-01-08,17091.144531,0.008025,1.028034,NaN,NaN,NaN,17091.144531,0.000000,BTC-USD
8,2023-01-09,17196.554688,0.006168,1.034374,NaN,NaN,NaN,17196.554688,0.000000,BTC-USD
9,2023-01-10,17446.292969,0.014523,1.049396,NaN,NaN,NaN,17446.292969,0.000000,BTC-USD


In [ ]:
master.to_csv("prices_master_v2.csv", index=False)

In [ ]:
master.isna().sum()

,0
Date,0
close,0
daily_return,4
cum_return,0
volatility_30d,120
rolling_return_30d,120
rolling_return_90d,360
rolling_max,0
drawdown,0
asset,0


In [ ]:
summary = master.groupby("asset").agg(
    final_cum_return=("cum_return", "last"),
    avg_volatility_30d=("volatility_30d", "mean"),
    max_drawdown=("drawdown", "min"),
    latest_30d_return=("rolling_return_30d", "last"),
    latest_90d_return=("rolling_return_90d", "last")
).reset_index()

summary

,asset,final_cum_return,avg_volatility_30d,max_drawdown,latest_30d_return,latest_90d_return
0,BTC-USD,4.039809,0.024011,-0.497388,-0.076308,-0.258722
1,ETH-USD,1.721725,0.032481,-0.637877,-0.027639,-0.338520
2,QQQ,2.254530,0.011585,-0.227683,-0.029417,0.001370
3,SPY,1.796440,0.008606,-0.187552,-0.039240,0.010780


In [ ]:
# ⚠️ Colab only — mount Google Drive
# 載入 Google Drive 連接工具
from google.colab import drive

# 掛載 Google Drive 到 Colab 環境
drive.mount("/content/drive")

In [4]:
# 載入必要套件
import pandas as pd
import yfinance as yf
import os

# 設定 Google Drive 裡的專案資料夾路徑
project_path = "/content/drive/MyDrive/global_asset_dashboard"

# 如果資料夾不存在就自動建立
os.makedirs(project_path, exist_ok=True)

print("資料夾已建立：", project_path)

資料夾已建立： /content/drive/MyDrive/global_asset_dashboard


In [5]:
# 設定要抓取的四個資產代號
tickers = ["BTC-USD", "ETH-USD", "SPY", "QQQ"]

# 建立一個空的 list，之後用來收集每個資產的資料
all_data = []

# 用 for 迴圈逐一抓取每個資產的歷史價格
for ticker in tickers:
    # 從 Yahoo Finance 抓取歷史價格資料
    df = yf.download(ticker, start="2020-01-01", end="2025-12-31", auto_adjust=True)

    # 只保留收盤價欄位
    df = df[["Close"]].copy()

    # 把欄位名稱改成小寫
    df.columns = ["close"]

    # 新增一欄記錄這是哪個資產
    df["asset"] = ticker

    # 把 index 名稱改成 date
    df.index.name = "date"

    # 重設 index，讓 date 變成一般欄位
    df = df.reset_index()

    # 計算日報酬率
    df["daily_return"] = df["close"].pct_change()

    # 計算累積報酬率，第一天填 1 作為基準
    df["cum_return"] = (1 + df["daily_return"]).cumprod().fillna(1)

    # 把這個資產的資料放進 all_data
    all_data.append(df)

# 把四個資產的資料合併成一張大表
prices_master = pd.concat(all_data, ignore_index=True)

# 存到 Google Drive
prices_master.to_csv(f"{project_path}/prices_master.csv", index=False)

print("prices_master 筆數：", prices_master.shape)
print("prices_master.csv 已儲存到 Google Drive")

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

prices_master 筆數： (7396, 5)
prices_master.csv 已儲存到 Google Drive


In [6]:
# 計算 30 日波動率，按資產分組計算
prices_master["volatility_30d"] = prices_master.groupby("asset")["daily_return"].transform(
    lambda x: x.rolling(30).std()
)

# 定義計算 CAGR 的函數
def calc_cagr(group):
    # 取第一天和最後一天的收盤價
    start = group["close"].iloc[0]
    end = group["close"].iloc[-1]

    # 計算總天數（日曆天）
    days = (pd.to_datetime(group["date"].iloc[-1]) - pd.to_datetime(group["date"].iloc[0])).days

    # 換算成年數
    years = days / 365

    # 套用 CAGR 公式
    cagr = (end / start) ** (1 / years) - 1
    return cagr

# 定義計算 Max Drawdown 的函數
def calc_max_drawdown(group):
    # 計算歷史最高累積報酬
    cummax = group["cum_return"].cummax()

    # 計算每天相對最高點的跌幅
    drawdown = (group["cum_return"] - cummax) / cummax

    # 取最大跌幅
    return drawdown.min()

# 對每個資產計算 CAGR
cagr_df = prices_master.groupby("asset").apply(calc_cagr, include_groups=False).reset_index()
cagr_df.columns = ["asset", "cagr"]

# 對每個資產計算 Max Drawdown
mdd_df = prices_master.groupby("asset").apply(calc_max_drawdown, include_groups=False).reset_index()
mdd_df.columns = ["asset", "max_drawdown"]

# 計算每個資產的平均波動率
vol_df = prices_master.groupby("asset")["volatility_30d"].mean().reset_index()
vol_df.columns = ["asset", "avg_volatility_30d"]

# 把三個指標合併成一張表
metrics = cagr_df.merge(mdd_df, on="asset").merge(vol_df, on="asset")

# 存到 Google Drive
metrics.to_csv(f"{project_path}/metrics.csv", index=False)

print(metrics)
print("metrics.csv 已儲存到 Google Drive")

     asset      cagr  max_drawdown  avg_volatility_30d
0  BTC-USD  0.518948     -0.766346            0.029542
1  ETH-USD  0.682900     -0.793512            0.039689
2      QQQ  0.199009     -0.351187            0.014406
3      SPY  0.149631     -0.337173            0.011136
metrics.csv 已儲存到 Google Drive


In [12]:
# 載入套件
import requests
import pandas as pd

# Fear & Greed Index 公開 API，limit=2000 代表抓最近 2000 天的資料
url = "https://api.alternative.me/fng/?limit=2000"

# 發送請求
response = requests.get(url)

# 印出狀態碼確認連線成功
print("狀態碼：", response.status_code)

# 印出前 300 個字看看資料結構
print(response.text[:300])

狀態碼： 200
{
	"name": "Fear and Greed Index",
	"data": [
		{
			"value": "13",
			"value_classification": "Extreme Fear",
			"timestamp": "1775433600",
			"time_until_update": "60247"
		},
		{
			"value": "12",
			"value_classification": "Extreme Fear",
			"timestamp": "1775347200"
		},
		{
			"value": "11",
	


In [14]:
# 把 JSON 資料轉成 Python 字典
data = response.json()

# 取出 data 欄位裡的列表
records = data["data"]

# 轉成 DataFrame
fng_df = pd.DataFrame(records)

# 把 Unix 時間戳記轉成日期格式
fng_df["date"] = pd.to_datetime(fng_df["timestamp"].astype(int), unit="s")

# 只保留需要的欄位
fng_df = fng_df[["date", "value", "value_classification"]]

# 把 value 欄位從文字轉成數字
fng_df["value"] = fng_df["value"].astype(int)

# 重新命名欄位
fng_df.columns = ["date", "fng_value", "fng_classification"]

# 按日期排序（從舊到新）
fng_df = fng_df.sort_values("date").reset_index(drop=True)

# 印出結果
print(fng_df.shape)
print(fng_df.head())
print(fng_df.tail())

# 存到 Google Drive
fng_df.to_csv(f"{project_path}/fear_greed.csv", index=False)
print("fear_greed.csv 已儲存到 Google Drive")

(2983, 3)
        date  fng_value fng_classification
0 2018-02-01         30               Fear
1 2018-02-02         15       Extreme Fear
2 2018-02-03         40               Fear
3 2018-02-04         24       Extreme Fear
4 2018-02-05         11       Extreme Fear
           date  fng_value fng_classification
2978 2026-04-02         12       Extreme Fear
2979 2026-04-03          9       Extreme Fear
2980 2026-04-04         11       Extreme Fear
2981 2026-04-05         12       Extreme Fear
2982 2026-04-06         13       Extreme Fear
fear_greed.csv 已儲存到 Google Drive


In [15]:
# 確認 prices_master 的日期範圍
print("prices_master 日期範圍：")
print("開始：", prices_master["date"].min())
print("結束：", prices_master["date"].max())

# 確認 fear_greed 的日期範圍
print("\nfear_greed 日期範圍：")
print("開始：", fng_df["date"].min())
print("結束：", fng_df["date"].max())

prices_master 日期範圍：
開始： 2020-01-01 00:00:00
結束： 2025-12-30 00:00:00

fear_greed 日期範圍：
開始： 2018-02-01 00:00:00
結束： 2026-04-06 00:00:00


In [16]:
# 設定對齊的時間範圍（以 prices_master 為基準）
start_date = "2020-01-01"
end_date = "2025-12-30"

# 裁切 fear_greed，只保留這個範圍內的資料
fng_df = fng_df[
    (fng_df["date"] >= start_date) &
    (fng_df["date"] <= end_date)
].reset_index(drop=True)

# 確認裁切後的結果
print("裁切後筆數：", fng_df.shape)
print("開始：", fng_df["date"].min())
print("結束：", fng_df["date"].max())

# 覆蓋存檔
fng_df.to_csv(f"{project_path}/fear_greed.csv", index=False)
print("fear_greed.csv 已更新")

裁切後筆數： (2190, 3)
開始： 2020-01-01 00:00:00
結束： 2025-12-30 00:00:00
fear_greed.csv 已更新


In [18]:
# 用 yfinance 抓 VIX 歷史資料
vix = yf.download("^VIX", start="2020-01-01", end="2025-12-31", auto_adjust=True)

# 只保留收盤價
vix_df = vix[["Close"]].copy()

# 重新命名欄位
vix_df.columns = ["vix_close"]

# 重設 index，讓日期變成一般欄位
vix_df.index.name = "date"
vix_df = vix_df.reset_index()

# 確認資料
print(vix_df.shape)
print(vix_df.head())

# 存到 Google Drive
vix_df.to_csv(f"{project_path}/vix.csv", index=False)
print("vix.csv 已儲存到 Google Drive")

[*********************100%***********************]  1 of 1 completed

(1507, 2)
        date  vix_close
0 2020-01-02      12.47
1 2020-01-03      14.02
2 2020-01-06      13.85
3 2020-01-07      13.79
4 2020-01-08      13.45
vix.csv 已儲存到 Google Drive


In [21]:
# ⚠️ Colab only — Google Cloud authentication
# 載入 Google Cloud 認證工具
from google.colab import auth

# 進行身份驗證，會跳出授權視窗
auth.authenticate_user()

print("認證完成")

認證完成


In [23]:
# 載入 BigQuery 套件
from google.cloud import bigquery
import pandas as pd

# 設定你的專案 ID（從 BigQuery 頁面左側複製）
project_id = "global-asset-dashboard-492507"

# 建立 BigQuery 客戶端，用來和 BigQuery 溝通
client = bigquery.Client(project=project_id)

# 設定資料集名稱
dataset_id = "portfolio"

print("BigQuery 連線成功")

BigQuery 連線成功


In [24]:
# 設定要上傳的檔案和對應的資料表名稱
tables = {
    "prices": f"{project_path}/prices_master.csv",
    "metrics": f"{project_path}/metrics.csv",
    "fear_greed": f"{project_path}/fear_greed.csv",
    "vix": f"{project_path}/vix.csv",
}

# 逐一上傳每個檔案
for table_name, file_path in tables.items():
    # 讀取 CSV 檔案
    df = pd.read_csv(file_path)

    # 設定目標資料表的完整路徑
    table_ref = f"{project_id}.{dataset_id}.{table_name}"

    # 設定上傳選項，如果資料表已存在就覆蓋
    job_config = bigquery.LoadJobConfig(
        write_disposition="WRITE_TRUNCATE",
        autodetect=True,
    )

    # 執行上傳
    job = client.load_table_from_dataframe(df, table_ref, job_config=job_config)

    # 等待上傳完成
    job.result()

    print(f"{table_name} 上傳完成，共 {len(df)} 筆")

print("所有資料表上傳完成")

prices 上傳完成，共 7396 筆
metrics 上傳完成，共 4 筆
fear_greed 上傳完成，共 2190 筆
vix 上傳完成，共 1507 筆
所有資料表上傳完成


In [34]:
import time
from bs4 import BeautifulSoup
import requests
import pandas as pd

# 60 個關鍵事件的日期清單（從 BigQuery 查詢結果整理）
key_dates = [
    "2020-03-12", "2020-03-09", "2020-03-16", "2020-06-11", "2020-02-27",
    "2020-02-24", "2020-03-24", "2020-09-03", "2020-03-13", "2020-11-06",
    "2021-11-26", "2021-01-22", "2021-05-12", "2021-01-28", "2021-01-27",
    "2021-03-01", "2021-05-19", "2021-05-05", "2021-05-20", "2021-09-20",
    "2022-06-13", "2022-05-05", "2022-04-26", "2022-04-22", "2022-11-10",
    "2022-11-09", "2022-02-28", "2022-07-18", "2022-02-04", "2022-11-08",
    "2023-03-13", "2023-10-23", "2023-03-17", "2023-02-15", "2023-11-09",
    "2023-01-20", "2023-08-17", "2023-03-12", "2023-02-02", "2023-11-15",
    "2024-12-18", "2024-08-05", "2024-07-15", "2024-07-24", "2024-05-20",
    "2024-08-08", "2024-11-11", "2024-03-20", "2024-02-28", "2024-11-06",
    "2025-04-03", "2025-04-10", "2025-04-04", "2025-04-09", "2025-10-10",
    "2025-11-04", "2025-05-08", "2025-10-11", "2025-03-03", "2025-08-22"
]

# 設定 headers
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# 收集所有新聞
all_news = []

for date in key_dates:
    # 計算前後一天作為搜尋範圍
    from datetime import datetime, timedelta
    dt = datetime.strptime(date, "%Y-%m-%d")
    before = (dt + timedelta(days=1)).strftime("%Y-%m-%d")
    after = (dt - timedelta(days=1)).strftime("%Y-%m-%d")

    # 建立 Google News RSS 網址
    url = f"https://news.google.com/rss/search?q=bitcoin+after:{after}+before:{before}&hl=en-US&gl=US&ceid=US:en"

    try:
        # 發送請求
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, "xml")
        items = soup.find_all("item")

        # 只取前 3 則最相關的新聞
        for item in items[:3]:
            all_news.append({
                "event_date": date,
                "title": item.title.text,
                "pub_date": item.pubDate.text,
                "source": item.source.text if item.source else None,
                "link": item.link.text if item.link else None
            })

        print(f"{date}：找到 {len(items)} 則，已存 3 則")

        # 每次請求間隔 1 秒，避免被擋
        time.sleep(1)

    except Exception as e:
        print(f"{date}：發生錯誤 - {e}")

# 轉成 DataFrame
news_df = pd.DataFrame(all_news)
print(f"總共抓到 {len(news_df)} 則新聞")
print(news_df.head())

2020-03-12：找到 34 則，已存 3 則
2020-03-09：找到 25 則，已存 3 則
2020-03-16：找到 17 則，已存 3 則
2020-06-11：找到 15 則，已存 3 則
2020-02-27：找到 25 則，已存 3 則
2020-02-24：找到 20 則，已存 3 則
2020-03-24：找到 14 則，已存 3 則
2020-09-03：找到 17 則，已存 3 則
2020-03-13：找到 26 則，已存 3 則
2020-11-06：找到 41 則，已存 3 則
2021-11-26：找到 21 則，已存 3 則
2021-01-22：找到 42 則，已存 3 則
2021-05-12：找到 100 則，已存 3 則
2021-01-28：找到 79 則，已存 3 則
2021-01-27：找到 75 則，已存 3 則
2021-03-01：找到 61 則，已存 3 則
2021-05-19：找到 100 則，已存 3 則
2021-05-05：找到 83 則，已存 3 則
2021-05-20：找到 100 則，已存 3 則
2021-09-20：找到 51 則，已存 3 則
2022-06-13：找到 100 則，已存 3 則
2022-05-05：找到 88 則，已存 3 則
2022-04-26：找到 87 則，已存 3 則
2022-04-22：找到 57 則，已存 3 則
2022-11-10：找到 75 則，已存 3 則
2022-11-09：找到 88 則，已存 3 則
2022-02-28：找到 69 則，已存 3 則
2022-07-18：找到 44 則，已存 3 則
2022-02-04：找到 57 則，已存 3 則
2022-11-08：找到 94 則，已存 3 則
2023-03-13：找到 41 則，已存 3 則
2023-10-23：找到 100 則，已存 3 則
2023-03-17：找到 47 則，已存 3 則
2023-02-15：找到 87 則，已存 3 則
2023-11-09：找到 68 則，已存 3 則
2023-01-20：找到 53 則，已存 3 則
2023-08-17：找到 78 則，已存 3 則
2023-03-12：找到 34 則，已存 3 則
2023-02

In [35]:
# 存到 Google Drive
news_df.to_csv(f"{project_path}/key_events_news.csv", index=False)
print("key_events_news.csv 已儲存到 Google Drive")

# 存到 BigQuery
table_ref = f"{project_id}.{dataset_id}.key_events_news"

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE",
    autodetect=True,
)

job = client.load_table_from_dataframe(news_df, table_ref, job_config=job_config)
job.result()

print("key_events_news 資料表已上傳到 BigQuery")
print(f"共 {len(news_df)} 則新聞")

key_events_news.csv 已儲存到 Google Drive
key_events_news 資料表已上傳到 BigQuery
共 180 則新聞


In [36]:
# 從 BigQuery 讀取四個資料表，存成 CSV 到 Google Drive

# 設定要匯出的資料表
tables_to_export = {
    "prices": "SELECT * FROM `global-asset-dashboard-492507.portfolio.prices`",
    "metrics": "SELECT * FROM `global-asset-dashboard-492507.portfolio.metrics`",
    "key_events": "SELECT * FROM `global-asset-dashboard-492507.portfolio.key_events`",
    "key_events_news": "SELECT * FROM `global-asset-dashboard-492507.portfolio.key_events_news`"
}

# 逐一匯出
for table_name, query in tables_to_export.items():
    # 執行查詢
    df = client.query(query).to_dataframe()

    # 存到 Google Drive
    file_path = f"{project_path}/{table_name}_export.csv"
    df.to_csv(file_path, index=False)

    print(f"{table_name}_export.csv 已儲存，共 {len(df)} 筆")

print("\n所有資料表匯出完成")

prices_export.csv 已儲存，共 7396 筆
metrics_export.csv 已儲存，共 4 筆
key_events_export.csv 已儲存，共 60 筆
key_events_news_export.csv 已儲存，共 180 筆

所有資料表匯出完成


In [37]:
# 從 BigQuery 讀取 key_events，並把對應的新聞標題合併進來
query = """
SELECT
    k.date,
    k.year,
    k.rank,
    k.anomaly_count,
    k.anomaly_types,
    k.severity_score,
    k.total_score,
    -- 把同一天的三則新聞標題合併成一個欄位
    STRING_AGG(n.title, ' | ' ORDER BY n.title) AS news_titles,
    STRING_AGG(n.source, ' | ' ORDER BY n.source) AS news_sources
FROM `global-asset-dashboard-492507.portfolio.key_events` AS k
LEFT JOIN `global-asset-dashboard-492507.portfolio.key_events_news` AS n
    ON k.date = DATE(n.event_date)
GROUP BY k.date, k.year, k.rank, k.anomaly_count, k.anomaly_types, k.severity_score, k.total_score
ORDER BY k.year ASC, k.rank ASC
"""

# 執行查詢
key_events_with_news = client.query(query).to_dataframe()

# 確認結果
print(key_events_with_news.shape)
print(key_events_with_news[["date", "anomaly_types", "news_titles"]].head())

# 存到 Google Drive
key_events_with_news.to_csv(f"{project_path}/key_events_with_news.csv", index=False)
print("key_events_with_news.csv 已儲存")

(60, 9)
         date        anomaly_types  \
0  2020-03-12         VIX異常 + 價格異常   
1  2020-03-09  VIX異常 + 價格異常 + 情緒異常   
2  2020-03-16         VIX異常 + 價格異常   
3  2020-06-11         VIX異常 + 價格異常   
4  2020-02-27         VIX異常 + 價格異常   

                                         news_titles  
0  A percolation model for the emergence of the B...  
1  French Court Says Bitcoin Is Money - PYMNTS.co...  
2  Bitcoin Is Now Undervalued, Suggests This Pric...  
3  Bitcoin scammers take YouTube channels for a S...  
4  How to Invest in Bitcoin with a Trust - Bitwis...  
key_events_with_news.csv 已儲存
